In [2]:
import numpy as np

def generate_disk_dataset(
    n_samples=200,
    image_size=128,
    radius_min=10,       # 圆盘半径范围
    radius_max=40,
    center_std=5.0,      # 中心偏移标准差
    dtype=np.float32
):
    """
    Generate a solid circular disk dataset with anti-aliasing and randomized centers.
    """
    dataset = np.zeros(
        (n_samples, 1, image_size, image_size),
        dtype=dtype
    )

    y, x = np.indices((image_size, image_size))
    base_center = (image_size - 1) / 2.0

    for i in range(n_samples):
        # 1. 随机中心点
        cx = np.random.normal(loc=base_center, scale=center_std)
        cy = np.random.normal(loc=base_center, scale=center_std)

        # 2. 计算距离矩阵
        r = np.sqrt((x - cx)**2 + (y - cy)**2)

        # 3. 随机生成圆盘半径
        radius = np.random.uniform(radius_min, radius_max)

        # 4. 软边缘处理 (Anti-aliasing)
        # 因为是实心盘，只需要外边缘的截断，不需要 inner_mask
        disk = np.clip(radius - r + 0.5, 0.0, 1.0).astype(dtype)

        dataset[i, 0] = disk

    return dataset


# ==================================================
# Example
# ==================================================
dataset_disks = generate_disk_dataset(
    n_samples=200,
    image_size=128,
    radius_min=3,
    radius_max=24,
    center_std=12.0
)

print("Disk dataset shape:", dataset_disks.shape)
np.save("disk_dataset.npy", dataset_disks)

Disk dataset shape: (200, 1, 128, 128)


In [3]:
import numpy as np

def generate_perfect_elliptical_disk_dataset(
    n_samples=200,
    image_size=128,
    a_min=3, a_max=10,    # 适合微小椭圆
    b_min=3, b_max=10,
    center_std=12.0,
    dtype=np.float32
):
    """
    Generate a perfect solid elliptical disk dataset using gradient-based SDF anti-aliasing.
    """
    dataset = np.zeros(
        (n_samples, 1, image_size, image_size),
        dtype=dtype
    )

    y, x = np.indices((image_size, image_size))
    base_center = (image_size - 1) / 2.0

    for i in range(n_samples):
        # 1. 随机参数
        cx = np.random.normal(loc=base_center, scale=center_std)
        cy = np.random.normal(loc=base_center, scale=center_std)
        a = np.random.uniform(a_min, a_max)
        b = np.random.uniform(b_min, b_max)
        theta = np.random.uniform(0, np.pi)

        # 2. 坐标平移与旋转
        dx = x - cx
        dy = y - cy
        x_rot = dx * np.cos(theta) + dy * np.sin(theta)
        y_rot = -dx * np.sin(theta) + dy * np.cos(theta)

        # ==================================================
        # 核心优化：基于梯度的距离场计算 (Gradient-based SDF)
        # ==================================================
        
        # 3. 计算椭圆的隐函数值 F(x,y)
        # F < 0 表示在椭圆内部，F > 0 表示在外部，F = 0 在边界上
        f_val = (x_rot / a)**2 + (y_rot / b)**2 - 1.0

        # 4. 计算梯度 (Gradient) 的大小
        # 梯度表示隐函数值在空间中变化的最快方向和速率
        grad_x = 2 * x_rot / (a**2)
        grad_y = 2 * y_rot / (b**2)
        grad_mag = np.sqrt(grad_x**2 + grad_y**2) + 1e-8  # 加上极小值防止中心点除以 0

        # 5. 估算每个像素到椭圆边界的精确垂直距离
        # 将无量纲的 f_val 转化为实际的像素距离
        dist = f_val / grad_mag

        # 6. 软边缘处理
        # dist <= 0 在内部，dist > 0 在外部。
        # 0.5 - dist 会在边界线 +/- 0.5 像素范围内形成 0 到 1 的完美渐变
        elliptical_disk = np.clip(0.5 - dist, 0.0, 1.0).astype(dtype)
        
        dataset[i, 0] = elliptical_disk

    return dataset


# ==================================================
# Example
# ==================================================
dataset_elliptical_disks = generate_perfect_elliptical_disk_dataset(
    n_samples=200,
    image_size=128,
    a_min=3, a_max=20,
    b_min=6, b_max=16,
    center_std=12.0
)

print("Perfect elliptical disk dataset shape:", dataset_elliptical_disks.shape)
np.save("elliptical_disk_dataset.npy", dataset_elliptical_disks)

Perfect elliptical disk dataset shape: (200, 1, 128, 128)
